# CPSA 02 — build the full-frame YOLO-pose dataset

Inputs: the public COCO 2017 images dataset attached on Kaggle
(`awsaf49/coco-2017-dataset`) and the half-bust dataset built by notebook 01
(`fabiocabeccia/cpsa-halfbust`).  The COCO-WholeBody v1.0 annotations are
downloaded from the official Google Drive links
(https://github.com/jin-s13/COCO-WholeBody) into `/tmp/coco_wholebody`;
enable Internet for this notebook.  If the download is blocked, attach a
dataset with the two `coco_wholebody_*_v1.0.json` files instead: they are used
when present.

The full-frame set keeps the original COCO image and labels each person with
the true COCO box, so the model learns the real person extent (the runtime
uses box height / face width to tell `full_body` from `half_body`).  No
hand-over-face occlusion is applied here.  The training split is sized to
make full frames 25% of the mixed set:
`n_full = round(0.25 / 0.75 * n_half)`, where `n_half` is read from the
half-bust `stats.json`; the val split writes every qualifying image.

Set `COMMIT_SHA` below to the exact repository commit for this run.  Outputs
go to `/kaggle/working/fullframe62` and are re-published as a dataset version
before the session ends.  No GPU needed.

In [ ]:
import subprocess
import sys

# Pin the exact CPSA_2026 commit for a reproducible Kaggle run. The [sources]
# extra installs gdown for the annotation download.
COMMIT_SHA = "a06184367d62061955867d6199b949383e659763"

subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        f"cpsa-ml[sources] @ git+https://github.com/fcabecciaw/CPSA_2026@{COMMIT_SHA}#subdirectory=ml",
    ],
    check=True,
)

In [ ]:
from pathlib import Path

from cpsa_ml.datasets.sources import find_coco_root, wholebody_annotations
from cpsa_ml.datasets.staging import find_inputs

INPUT_DIR = Path("/kaggle/input")  # attached datasets
ANNOTATION_CACHE = Path("/tmp/coco_wholebody")  # outside /kaggle/working

COCO_ROOT = find_coco_root(INPUT_DIR)
WHOLEBODY = wholebody_annotations(INPUT_DIR, ANNOTATION_CACHE)
print("COCO root:", COCO_ROOT)
for split, path in WHOLEBODY.items():
    print(split, path)

In [ ]:
from cpsa_ml.datasets.fullframe import (
    FullFrameConfig,
    build_fullframe_dataset,
    half_train_count,
)

OUT = Path("/kaggle/working/fullframe62")
HALF_ROOT = next(  # attached notebook-01 dataset, located by its data.yaml
    p.parent for p in find_inputs(INPUT_DIR, "data.yaml") if "cpsa-halfbust" in str(p)
)
FRACTION = 0.25  # share of full frames in the mixed training set
SEED = 0

N_HALF = half_train_count(HALF_ROOT)
print("half-bust train images:", N_HALF)

for split in ("train", "val"):
    report = build_fullframe_dataset(
        FullFrameConfig(
            coco_root=COCO_ROOT,
            wholebody_json=WHOLEBODY[split],
            split=split,
            out=OUT,
            fraction=FRACTION,
            half_train_count=N_HALF if split == "train" else None,
            seed=SEED,
        )
    )
    print(split, report.to_dict())

In [ ]:
print((OUT / "data.yaml").read_text())
print((OUT / "stats.json").read_text())

## Next steps

Re-publish `/kaggle/working/fullframe62` as a new version of the
`fabiocabeccia/cpsa-fullframe` dataset, then attach both `cpsa-halfbust` and
`cpsa-fullframe` to notebook 03 (FP32 training) or notebook 04 (variant
study); those notebooks build the mixed `data.yaml` and report the half-body
(`val`) and full-frame (`val_full`) splits separately.